<a href="https://colab.research.google.com/github/kaluarachchiteshan/agromaster-risk-calculation-ml/blob/01_data_loading/notebooks/01_data_loading.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Google Drive connect
from google.colab import drive
drive.mount('/content/drive')

import os, glob, kagglehub
OUT_DIR = "/content/drive/MyDrive/agromaster"
os.makedirs(OUT_DIR, exist_ok=True)

# Dataset download from Kaggle (2 GB)
DATA_DIR = kagglehub.dataset_download("khandelwalmanas/daily-commodity-prices-india")
print("Data folder:", DATA_DIR)

files = sorted(glob.glob(os.path.join(DATA_DIR, "**", "*.parquet"), recursive=True))
print(len(files), "parquet files")
print(files[:2])

Mounted at /content/drive


100%|██████████| 1.41G/1.41G [00:10<00:00, 138MB/s]

Extracting files...


Data folder: /root/.cache/kagglehub/datasets/khandelwalmanas/daily-commodity-prices-india/versions/134
26 parquet files
['/root/.cache/kagglehub/datasets/khandelwalmanas/daily-commodity-prices-india/versions/134/parquet/2001.parquet', '/root/.cache/kagglehub/datasets/khandelwalmanas/daily-commodity-prices-india/versions/134/parquet/2002.parquet']


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Pass 1: coverage table - one row per series, development period only
import json, datetime, time
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

# ---------------- CONFIG ----------------
DEV_END = datetime.date(2023, 12, 31)   # end of development period (proposal 4.6)
MIN_DAYS_PER_SERIES = 1500              # reporting days a series needs in the development period
MIN_YEARS_SPAN = 8                      # first-to-last span a series needs in the development period
# ----------------------------------------
KEYS = ["state", "district", "market", "commodity", "variety", "grade"]   # one series = one combination
PRICES = ["min_price", "max_price", "modal_price"]
RENAME = {"State": "state", "District": "district", "Market": "market", "Commodity": "commodity",
          "Variety": "variety", "Grade": "grade", "Arrival_Date": "date",
          "Min_Price": "min_price", "Max_Price": "max_price", "Modal_Price": "modal_price"}

def read_year(path):
    """Read one yearly file as an Arrow table: standard names, trimmed text, date32 dates, float32 prices."""
    t = pq.read_table(path, columns=list(RENAME))
    cols = {}
    for src, std in RENAME.items():
        a = t[src]
        if std in KEYS:
            a = pc.utf8_trim_whitespace(pc.fill_null(a, "Unknown"))
        elif std == "date":
            a = pc.strptime(a, format="%Y-%m-%d", unit="s", error_is_null=True).cast(pa.date32())
        else:
            a = a.cast(pa.float32())
        cols[std] = a
    return pa.table(cols)

t0 = time.time()
parts, com_parts = [], []
total_rows = bad_dates = 0
dmin, dmax = datetime.date.max, datetime.date.min
for f in files:
    t = read_year(f)
    total_rows += t.num_rows; bad_dates += t["date"].null_count
    t = t.filter(pc.is_valid(t["date"]))
    mm = pc.min_max(t["date"]).as_py(); dmin, dmax = min(dmin, mm["min"]), max(dmax, mm["max"])
    com_parts.append(t.group_by("commodity").aggregate([("date", "count")]).to_pandas())
    dev = t.filter(pc.less_equal(t["date"], pa.scalar(DEV_END, pa.date32())))
    if dev.num_rows:
        parts.append(dev.group_by(KEYS).aggregate([("date", "count_distinct"), ("date", "min"), ("date", "max")]).to_pandas())
    print(os.path.basename(f), f"{t.num_rows:>10,} rows | total {total_rows:,} | {time.time() - t0:.0f}s")

cov = pd.concat(parts, ignore_index=True)
cov["date_min"] = pd.to_datetime(cov["date_min"]); cov["date_max"] = pd.to_datetime(cov["date_max"])
cov = (cov.groupby(KEYS, sort=False)
          .agg(n_days=("date_count_distinct", "sum"), first=("date_min", "min"), last=("date_max", "max")).reset_index())
cov["span_years"] = (cov["last"] - cov["first"]).dt.days / 365.25
cov["good"] = (cov["n_days"] >= MIN_DAYS_PER_SERIES) & (cov["span_years"] >= MIN_YEARS_SPAN)
cov.to_csv(os.path.join(OUT_DIR, "series_coverage_dev.csv"), index=False)

by_com = cov.groupby("commodity").agg(dev_rows=("n_days", "sum"), series=("n_days", "size"), good_series=("good", "sum"))
by_com["good_rows"] = cov[cov["good"]].groupby("commodity")["n_days"].sum()
by_com["good_rows"] = by_com["good_rows"].fillna(0).astype(int)
by_com["all_rows"] = pd.concat(com_parts).groupby("commodity")["date_count"].sum()
by_com = by_com.sort_values("good_rows", ascending=False)
by_com.to_csv(os.path.join(OUT_DIR, "commodity_coverage_dev.csv"))

print(f"\nSource rows: {total_rows:,} | bad dates: {bad_dates:,} | date range: {dmin} to {dmax}")
print(f"Commodities: {len(by_com)} | series in dev period: {len(cov):,} | good series: {int(cov['good'].sum()):,}")
print(by_com[["all_rows", "series", "good_series", "good_rows"]].head(15).to_string())

2001.parquet     11,603 rows | total 11,603 | 0s
2002.parquet    299,930 rows | total 311,533 | 1s
2003.parquet    835,030 rows | total 1,146,563 | 4s
2004.parquet    802,338 rows | total 1,948,901 | 6s
2005.parquet  1,073,632 rows | total 3,022,533 | 7s
2006.parquet  1,993,050 rows | total 5,015,583 | 10s
2007.parquet  2,378,705 rows | total 7,394,288 | 13s
2008.parquet  2,575,390 rows | total 9,969,678 | 16s
2009.parquet  2,483,396 rows | total 12,453,074 | 21s
2010.parquet  2,708,429 rows | total 15,161,503 | 24s
2011.parquet  2,899,187 rows | total 18,060,690 | 28s
2012.parquet  3,147,369 rows | total 21,208,059 | 32s
2013.parquet  3,274,568 rows | total 24,482,627 | 38s
2014.parquet  3,271,883 rows | total 27,754,510 | 41s
2015.parquet  3,673,823 rows | total 31,428,333 | 46s
2016.parquet  3,634,455 rows | total 35,062,788 | 52s
2017.parquet  4,089,861 rows | total 39,152,649 | 60s
2018.parquet  4,285,826 rows | total 43,438,475 | 75s
2019.parquet  4,319,450 rows | total 47,757,92

In [ ]:
# Pass 2: choose the modelling subset, extract it, save as compressed Parquet
COMMODITIES = ["Tomato", "Onion", "Potato", "Brinjal", "Green Chilli"]   # perishable vegetables with the best coverage
missing = [c for c in COMMODITIES if c not in by_com.index]
assert not missing, missing
keep = cov.loc[cov["good"] & cov["commodity"].isin(COMMODITIES), KEYS]

t0 = time.time(); kept = []
for f in files:
    t = read_year(f)
    t = t.filter(pc.is_valid(t["date"]))
    t = t.filter(pc.is_in(t["commodity"], value_set=pa.array(COMMODITIES)))
    kept.append(t.to_pandas().merge(keep, on=KEYS, how="inner"))
sub = pd.concat(kept, ignore_index=True)
sub["date"] = pd.to_datetime(sub["date"])
sub = sub.sort_values(KEYS + ["date"], kind="stable").reset_index(drop=True)
for c in KEYS:                      # categorical codes instead of repeated strings
    sub[c] = sub[c].astype("category")
out = os.path.join(OUT_DIR, "agmarknet_subset.parquet")
sub.to_parquet(out, compression="zstd", index=False)

dev_mask = sub["date"] <= pd.Timestamp(DEV_END)
summary = {
    "source_rows_total": int(total_rows),
    "source_date_range": [str(dmin), str(dmax)],
    "subset_rows": int(len(sub)),
    "subset_date_range": [str(sub["date"].min().date()), str(sub["date"].max().date())],
    "commodities": COMMODITIES,
    "n_series": int(len(keep)),
    "n_markets": int(sub["market"].nunique()),
    "n_states": int(sub["state"].nunique()),
    "selection_rule": f">= {MIN_DAYS_PER_SERIES} reporting days and >= {MIN_YEARS_SPAN} years span up to {DEV_END}",
    "rows_development": int(dev_mask.sum()),
    "rows_test": int((~dev_mask).sum()),
    "file_mb": round(os.path.getsize(out) / 1e6, 1),
}
json.dump(summary, open(os.path.join(OUT_DIR, "subset_summary.json"), "w"), indent=2)
print(json.dumps(summary, indent=2))
print(sub.groupby("commodity", observed=True).agg(rows=("date", "size"), markets=("market", "nunique"), states=("state", "nunique")).to_string())
print(f"done in {time.time() - t0:.0f}s | saved to {out}")

{
  "source_rows_total": 75984017,
  "source_date_range": [
    "2001-01-10",
    "2026-04-21"
  ],
  "subset_rows": 7919229,
  "subset_date_range": [
    "2001-01-10",
    "2025-11-06"
  ],
  "commodities": [
    "Tomato",
    "Onion",
    "Potato",
    "Brinjal",
    "Green Chilli"
  ],
  "n_series": 2605,
  "n_markets": 787,
  "n_states": 25,
  "selection_rule": ">= 1500 reporting days and >= 8 years span up to 2023-12-31",
  "rows_development": 7135630,
  "rows_test": 783599,
  "file_mb": 32.9
}
                 rows  markets  states
commodity                             
Brinjal       1337605      427      21
Green Chilli  1181890      380      23
Onion         1937749      566      23
Potato        1925794      571      25
Tomato        1536191      476      21
done in 85s | saved to /content/drive/MyDrive/agromaster/agmarknet_subset.parquet
